In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/plant-leaves-super-resolution-challenge/sample_submission.csv
/kaggle/input/competitions/plant-leaves-super-resolution-challenge/vgg19_weights.pth
/kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution/agrivision_train_0772.png
/kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution/agrivision_train_0802.png
/kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution/agrivision_train_1205.png
/kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution/agrivision_train_1566.png
/kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution/agrivision_train_0518.png
/kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution/agrivision_train_0276.png
/kaggle/input/competitions/plant-leaves-super-resolution-challenge/train_Low_Resolution/agrivision_train_0497.png
/kaggle/input/competitions/p

**Importing libraries**

In [2]:
from PIL import Image
import torch
from tqdm import tqdm
import random
import gc

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import vgg19

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [3]:
DATA_ROOT = "/kaggle/input/competitions/plant-leaves-super-resolution-challenge"

print("Check folder:", os.listdir(DATA_ROOT))

TRAIN_HR_DIR = os.path.join(DATA_ROOT, "train_High_Resolution")
TRAIN_LR_DIR = os.path.join(DATA_ROOT, "train_Low_Resolution")
TEST_LR_DIR  = os.path.join(DATA_ROOT, "test_Low_Resolution")
VGG_WEIGHTS  = os.path.join(DATA_ROOT, "vgg19_weights.pth")

print("LR exists:", os.path.exists(TRAIN_LR_DIR))
print("HR exists:", os.path.exists(TRAIN_HR_DIR))
print("Test exists:", os.path.exists(TEST_LR_DIR))

Check folder: ['sample_submission.csv', 'train_Low_Resolution', 'test_Low_Resolution', 'vgg19_weights.pth', 'train_High_Resolution']
LR exists: True
HR exists: True
Test exists: True


**Hyperparameters**

In [4]:
BATCH_SIZE = 16
NUM_EPOCHS = 120
LR_G = 2e-4
LR_D = 1e-4
LAMBDA_L1 = 100.0 
LAMBDA_PERC = 1.0      
LAMBDA_ADV = 0.05      
SCALE = 4

In [5]:
class SRDataset(Dataset):
    def __init__(self, lr_dir, hr_dir=None, augment=False):
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.files = sorted(os.listdir(lr_dir))
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def _to_tensor(self, img):
        # [0,255] -> [-1,1]
        arr = np.array(img, dtype=np.float32) / 127.5 - 1.0
        return torch.from_numpy(arr).permute(2, 0, 1).contiguous()

    def __getitem__(self, idx):
        fname = self.files[idx]
        lr = Image.open(os.path.join(self.lr_dir, fname)).convert("RGB")
        if self.hr_dir is not None:
            hr = Image.open(os.path.join(self.hr_dir, fname)).convert("RGB")
        else:
            hr = None

        if self.augment and hr is not None:
            # Random horizontal flip
            if random.random() < 0.5:
                lr = lr.transpose(Image.FLIP_LEFT_RIGHT)
                hr = hr.transpose(Image.FLIP_LEFT_RIGHT)
            # Random vertical flip
            if random.random() < 0.5:
                lr = lr.transpose(Image.FLIP_TOP_BOTTOM)
                hr = hr.transpose(Image.FLIP_TOP_BOTTOM)
            # Random 90 rotations
            k = random.randint(0, 3)
            if k > 0:
                lr = lr.rotate(90 * k)
                hr = hr.rotate(90 * k)

        lr_t = self._to_tensor(lr)
        if hr is not None:
            hr_t = self._to_tensor(hr)
            return lr_t, hr_t, fname
        return lr_t, fname


train_ds = SRDataset(TRAIN_LR_DIR, TRAIN_HR_DIR, augment=True)
test_ds  = SRDataset(TEST_LR_DIR, None, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False,
                          num_workers=2, pin_memory=True)

print("Train size:", len(train_ds), "| Test size:", len(test_ds))

Train size: 1642 | Test size: 495


**Generator (RRDB-style, randomly initialized)**

In [6]:
class ResidualDenseBlock(nn.Module):
    def __init__(self, nf=64, gc=32):
        super().__init__()
        self.conv1 = nn.Conv2d(nf,         gc, 3, 1, 1)
        self.conv2 = nn.Conv2d(nf + gc,    gc, 3, 1, 1)
        self.conv3 = nn.Conv2d(nf + 2*gc,  gc, 3, 1, 1)
        self.conv4 = nn.Conv2d(nf + 3*gc,  gc, 3, 1, 1)
        self.conv5 = nn.Conv2d(nf + 4*gc,  nf, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat([x, x1], 1)))
        x3 = self.lrelu(self.conv3(torch.cat([x, x1, x2], 1)))
        x4 = self.lrelu(self.conv4(torch.cat([x, x1, x2, x3], 1)))
        x5 = self.conv5(torch.cat([x, x1, x2, x3, x4], 1))
        return x5 * 0.2 + x


class RRDB(nn.Module):
    def __init__(self, nf=64, gc=32):
        super().__init__()
        self.b1 = ResidualDenseBlock(nf, gc)
        self.b2 = ResidualDenseBlock(nf, gc)
        self.b3 = ResidualDenseBlock(nf, gc)

    def forward(self, x):
        out = self.b1(x)
        out = self.b2(out)
        out = self.b3(out)
        return out * 0.2 + x


class Generator(nn.Module):
    def __init__(self, nf=64, nb=12, gc=32, scale=4):
        super().__init__()
        self.conv_first = nn.Conv2d(3, nf, 3, 1, 1)
        self.body = nn.Sequential(*[RRDB(nf, gc) for _ in range(nb)])
        self.conv_body = nn.Conv2d(nf, nf, 3, 1, 1)

        # Upsampling x4 via two PixelShuffle stages
        self.up1 = nn.Conv2d(nf, nf * 4, 3, 1, 1)
        self.up2 = nn.Conv2d(nf, nf * 4, 3, 1, 1)
        self.pixel_shuffle = nn.PixelShuffle(2)

        self.conv_hr  = nn.Conv2d(nf, nf, 3, 1, 1)
        self.conv_last = nn.Conv2d(nf, 3, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, a=0.2, mode='fan_in')
                m.weight.data *= 0.1
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        fea = self.conv_first(x)
        trunk = self.conv_body(self.body(fea))
        fea = fea + trunk

        fea = self.lrelu(self.pixel_shuffle(self.up1(fea)))
        fea = self.lrelu(self.pixel_shuffle(self.up2(fea)))

        fea = self.lrelu(self.conv_hr(fea))
        out = self.conv_last(fea)
        return torch.tanh(out)

**Discriminator (PatchGAN, randomly initialized)**

In [7]:
class Discriminator(nn.Module):
    def __init__(self, in_ch=3, nf=64):
        super().__init__()
        def block(in_c, out_c, stride=2, bn=True):
            layers = [nn.Conv2d(in_c, out_c, 4, stride, 1)]
            if bn:
                layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.net = nn.Sequential(
            *block(in_ch, nf,       stride=2, bn=False),
            *block(nf,    nf * 2,   stride=2),
            *block(nf*2,  nf * 4,   stride=2),
            *block(nf*4,  nf * 8,   stride=1),
            nn.Conv2d(nf * 8, 1, 4, 1, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0.0, 0.02)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.normal_(m.weight, 1.0, 0.02)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

**VGG Perceptual Loss (loads provided weights offline)**

In [8]:
class VGGPerceptual(nn.Module):
    def __init__(self, weights_path, layer_idx=35):
        super().__init__()
        vgg = vgg19(weights=None)
        state = torch.load(weights_path, map_location='cpu')
        try:
            vgg.load_state_dict(state)
        except Exception:
            vgg.features.load_state_dict(state, strict=False)

        self.features = nn.Sequential(*list(vgg.features.children())[:layer_idx]).eval()
        for p in self.features.parameters():
            p.requires_grad = False

        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1))

    def _prep(self, x):
        
        x = (x + 1.0) / 2.0
        x = (x - self.mean) / self.std
        return x

    def forward(self, pred, target):
        f_pred = self.features(self._prep(pred))
        f_targ = self.features(self._prep(target))
        return F.l1_loss(f_pred, f_targ)


try:
    vgg_loss = VGGPerceptual(VGG_WEIGHTS).to(device)
    USE_PERCEPTUAL = True
    print("VGG perceptual loss loaded.")
except Exception as e:
    print("Could not load VGG weights, disabling perceptual loss:", e)
    USE_PERCEPTUAL = False

VGG perceptual loss loaded.


**Build Models, Optimizers, Schedulers**

In [9]:
G = Generator(nf=64, nb=12, gc=32, scale=4).to(device)
D = Discriminator().to(device)

opt_G = torch.optim.Adam(G.parameters(), lr=LR_G, betas=(0.9, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=LR_D, betas=(0.9, 0.999))

sched_G = torch.optim.lr_scheduler.CosineAnnealingLR(opt_G, T_max=NUM_EPOCHS, eta_min=1e-6)
sched_D = torch.optim.lr_scheduler.CosineAnnealingLR(opt_D, T_max=NUM_EPOCHS, eta_min=1e-6)

bce = nn.BCEWithLogitsLoss()
l1  = nn.L1Loss()

# Count params sanity
n_g = sum(p.numel() for p in G.parameters())
n_d = sum(p.numel() for p in D.parameters())
print(f"Generator params: {n_g/1e6:.2f}M | Discriminator params: {n_d/1e6:.2f}M")

Generator params: 9.01M | Discriminator params: 2.77M


**Warm-up Pretraining (L1 only, crucial for MAE)**

In [10]:
WARMUP_EPOCHS = 30

print("L1 WARMUP PHASE")
G.train()
for epoch in range(WARMUP_EPOCHS):
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Warmup {epoch+1}/{WARMUP_EPOCHS}")
    for lr_img, hr_img, _ in pbar:
        lr_img = lr_img.to(device, non_blocking=True)
        hr_img = hr_img.to(device, non_blocking=True)

        opt_G.zero_grad(set_to_none=True)
        sr = G(lr_img)
        loss = l1(sr, hr_img)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
        opt_G.step()

        epoch_loss += loss.item()
        pbar.set_postfix(l1=f"{loss.item():.4f}")

    print(f"Warmup epoch {epoch+1} | avg L1: {epoch_loss/len(train_loader):.4f}")

torch.save(G.state_dict(), "/kaggle/working/G_warmup.pth")
print("Warmup done, saved.")

L1 WARMUP PHASE


Warmup 1/30: 100%|██████████| 102/102 [00:29<00:00,  3.45it/s, l1=0.1807]


Warmup epoch 1 | avg L1: 0.2161


Warmup 2/30: 100%|██████████| 102/102 [00:26<00:00,  3.86it/s, l1=0.1708]


Warmup epoch 2 | avg L1: 0.1668


...

Warmup epoch 28 | avg L1: 0.1364


Warmup 29/30: 100%|██████████| 102/102 [00:28<00:00,  3.52it/s, l1=0.1510]


Warmup epoch 29 | avg L1: 0.1364


Warmup 30/30: 100%|██████████| 102/102 [00:28<00:00,  3.52it/s, l1=0.1238]

Warmup epoch 30 | avg L1: 0.1364
Warmup done, saved.


**Adversarial Training Phase**

In [11]:
print("GAN FINETUNING PHASE")
ADV_EPOCHS = NUM_EPOCHS - WARMUP_EPOCHS

for epoch in range(ADV_EPOCHS):
    G.train(); D.train()
    g_running, d_running, l1_running = 0.0, 0.0, 0.0
    pbar = tqdm(train_loader, desc=f"GAN {epoch+1}/{ADV_EPOCHS}")

    for lr_img, hr_img, _ in pbar:
        lr_img = lr_img.to(device, non_blocking=True)
        hr_img = hr_img.to(device, non_blocking=True)
        bsz = lr_img.size(0)

        # Train Discriminator 
        opt_D.zero_grad(set_to_none=True)
        with torch.no_grad():
            sr = G(lr_img)

        d_real = D(hr_img)
        d_fake = D(sr.detach())
        # Relativistic average GAN loss
        loss_d_real = bce(d_real - d_fake.mean(), torch.ones_like(d_real))
        loss_d_fake = bce(d_fake - d_real.mean(), torch.zeros_like(d_fake))
        loss_D = (loss_d_real + loss_d_fake) * 0.5
        loss_D.backward()
        torch.nn.utils.clip_grad_norm_(D.parameters(), 1.0)
        opt_D.step()

        # Train Generator
        opt_G.zero_grad(set_to_none=True)
        sr = G(lr_img)

        loss_l1 = l1(sr, hr_img)

        d_real = D(hr_img).detach()
        d_fake = D(sr)
        loss_adv = (bce(d_real - d_fake.mean(), torch.zeros_like(d_real)) +
                    bce(d_fake - d_real.mean(), torch.ones_like(d_fake))) * 0.5

        loss_G = LAMBDA_L1 * loss_l1 + LAMBDA_ADV * loss_adv
        if USE_PERCEPTUAL:
            loss_perc = vgg_loss(sr, hr_img)
            loss_G = loss_G + LAMBDA_PERC * loss_perc

        loss_G.backward()
        torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
        opt_G.step()

        g_running  += loss_G.item()
        d_running  += loss_D.item()
        l1_running += loss_l1.item()
        pbar.set_postfix(G=f"{loss_G.item():.3f}",
                         D=f"{loss_D.item():.3f}",
                         L1=f"{loss_l1.item():.4f}")

    sched_G.step(); sched_D.step()
    nb = len(train_loader)
    print(f"Epoch {epoch+1} | G {g_running/nb:.3f} | D {d_running/nb:.3f} | L1 {l1_running/nb:.4f}")

    # Periodic checkpoint
    if (epoch + 1) % 10 == 0:
        torch.save(G.state_dict(), f"/kaggle/working/G_epoch{epoch+1+WARMUP_EPOCHS}.pth")

torch.save(G.state_dict(), "/kaggle/working/G_final.pth")
print("Training complete.")

GAN FINETUNING PHASE


GAN 1/90: 100%|██████████| 102/102 [00:59<00:00,  1.73it/s, D=0.000, G=15.355, L1=0.1330]


Epoch 1 | G 16.031 | D 0.105 | L1 0.1396


GAN 2/90: 100%|██████████| 102/102 [00:55<00:00,  1.82it/s, D=0.000, G=17.440, L1=0.1536]


Epoch 2 | G 15.957 | D 0.000 | L1 0.1382


...

Epoch 88 | G 15.725 | D 0.000 | L1 0.1348


GAN 89/90: 100%|██████████| 102/102 [00:55<00:00,  1.83it/s, D=0.000, G=16.322, L1=0.1410]


Epoch 89 | G 15.718 | D 0.000 | L1 0.1348


GAN 90/90: 100%|██████████| 102/102 [00:55<00:00,  1.83it/s, D=0.000, G=16.901, L1=0.1474]

Epoch 90 | G 15.717 | D 0.000 | L1 0.1347
Training complete.


**Inference with Test-Time Augmentation (TTA**

In [12]:
G.eval()

@torch.no_grad()
def tta_infer(lr_batch):
    outs = []
    transforms_list = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[2]),
        lambda x: torch.flip(x, dims=[3]),
        lambda x: torch.flip(x, dims=[2, 3]),
    ]
    inv_transforms = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[2]),
        lambda x: torch.flip(x, dims=[3]),
        lambda x: torch.flip(x, dims=[2, 3]),
    ]
    for t, it in zip(transforms_list, inv_transforms):
        out = G(t(lr_batch))
        outs.append(it(out))
    return torch.stack(outs, dim=0).mean(dim=0)


predictions = {}
for lr_img, fnames in tqdm(test_loader, desc="Inference"):
    lr_img = lr_img.to(device, non_blocking=True)
    sr = tta_infer(lr_img)
    sr = ((sr + 1.0) * 127.5).clamp(0, 255).round().to(torch.uint8)
    sr = sr.permute(0, 2, 3, 1).cpu().numpy()
    for i, fn in enumerate(fnames):
        predictions[fn] = sr[i]

print("Predicted", len(predictions), "images.")

Inference: 100%|██████████| 62/62 [00:11<00:00,  5.37it/s]

Predicted 495 images.


In [13]:
rows = []
for fname in sorted(predictions.keys()):
    arr = predictions[fname]  
    flat = arr.reshape(-1)
    pixel_str = " ".join(flat.astype(str).tolist())
    rows.append({"Id": fname, "Pixels": pixel_str})

sub = pd.DataFrame(rows, columns=["Id", "Pixels"])
sub.to_csv("/kaggle/working/submission.csv", index=False)

print("Rows:", len(sub))
print("First id:", sub.iloc[0]["Id"])
print("Pixel count in first row:", len(sub.iloc[0]["Pixels"].split()))
assert len(sub.iloc[0]["Pixels"].split()) == 128*128*3, "Wrong pixel count!"
print("submission.csv ready")

Rows: 495
First id: agrivision_test_0000.png
Pixel count in first row: 49152
submission.csv ready
